In [ ]:
# ============================================================
# AI Based Intelligent RF Spectrum Signal Identification System
# Notebook: 04_cnn_model.ipynb
# Purpose: CNN Model Development for Modulation Classification
# ============================================================

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Flatten,
    Dense,
    Input
)
from tensorflow.keras.utils import to_categorical


# ============================================================
# 1. PROJECT INFORMATION
# ============================================================

print("=" * 70)
print("CNN MODEL DEVELOPMENT")
print("=" * 70)

print("\nTensorFlow Version:", tf.__version__)


# ============================================================
# 2. DEFINE PATHS
# ============================================================

processed_data_dir = "../data/processed"
models_dir = "../models"

os.makedirs(models_dir, exist_ok=True)

if not os.path.exists(processed_data_dir):
    raise FileNotFoundError(
        f"Processed data directory not found:\n"
        f"{os.path.abspath(processed_data_dir)}"
    )


# ============================================================
# 3. LOAD PROCESSED DATA
# ============================================================

print("\nLoading processed data...")

X_train = np.load(
    os.path.join(processed_data_dir, "X_train.npy")
)

X_val = np.load(
    os.path.join(processed_data_dir, "X_val.npy")
)

X_test = np.load(
    os.path.join(processed_data_dir, "X_test.npy")
)

y_train = np.load(
    os.path.join(processed_data_dir, "y_train.npy")
)

y_val = np.load(
    os.path.join(processed_data_dir, "y_val.npy")
)

y_test = np.load(
    os.path.join(processed_data_dir, "y_test.npy")
)

modulation_classes = np.load(
    os.path.join(
        processed_data_dir,
        "modulation_classes.npy"
    )
)

print("Processed data loaded successfully.")

print("\nOriginal Shapes:")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)


# ============================================================
# 4. GET NUMBER OF CLASSES
# ============================================================

num_classes = len(modulation_classes)

print("\nNumber of Modulation Classes:", num_classes)

print("\nModulation Classes:")

for index, modulation in enumerate(modulation_classes):
    print(f"{index}: {modulation}")


# ============================================================
# 5. RESHAPE DATA FOR CNN
# ============================================================

print("\n" + "=" * 70)
print("RESHAPING DATA FOR CNN")
print("=" * 70)

# Original shape:
# (samples, 2, 128)

# CNN shape:
# (samples, 2, 128, 1)

X_train_cnn = X_train[..., np.newaxis]
X_val_cnn = X_val[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]

print("\nCNN Input Shapes:")

print("X_train_cnn:", X_train_cnn.shape)
print("X_val_cnn:  ", X_val_cnn.shape)
print("X_test_cnn: ", X_test_cnn.shape)


# ============================================================
# 6. CONVERT LABELS TO CATEGORICAL FORMAT
# ============================================================

print("\n" + "=" * 70)
print("ENCODING OUTPUT LABELS")
print("=" * 70)

y_train_categorical = to_categorical(
    y_train,
    num_classes=num_classes
)

y_val_categorical = to_categorical(
    y_val,
    num_classes=num_classes
)

y_test_categorical = to_categorical(
    y_test,
    num_classes=num_classes
)

print("\nLabel Shapes:")

print("y_train:", y_train_categorical.shape)
print("y_val:  ", y_val_categorical.shape)
print("y_test: ", y_test_categorical.shape)


# ============================================================
# 7. BUILD CNN MODEL
# ============================================================

print("\n" + "=" * 70)
print("BUILDING CNN MODEL")
print("=" * 70)

model = Sequential([

    # Input Layer
    Input(shape=(2, 128, 1)),

    # --------------------------------------------------------
    # Convolution Block 1
    # --------------------------------------------------------

    Conv2D(
        filters=32,
        kernel_size=(1, 5),
        activation="relu",
        padding="same"
    ),

    BatchNormalization(),

    MaxPooling2D(
        pool_size=(1, 2)
    ),

    Dropout(0.20),


    # --------------------------------------------------------
    # Convolution Block 2
    # --------------------------------------------------------

    Conv2D(
        filters=64,
        kernel_size=(1, 3),
        activation="relu",
        padding="same"
    ),

    BatchNormalization(),

    MaxPooling2D(
        pool_size=(1, 2)
    ),

    Dropout(0.25),


    # --------------------------------------------------------
    # Convolution Block 3
    # --------------------------------------------------------

    Conv2D(
        filters=128,
        kernel_size=(1, 3),
        activation="relu",
        padding="same"
    ),

    BatchNormalization(),

    Dropout(0.30),


    # --------------------------------------------------------
    # Classification Layers
    # --------------------------------------------------------

    Flatten(),

    Dense(
        256,
        activation="relu"
    ),

    Dropout(0.50),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.30),


    # --------------------------------------------------------
    # Output Layer
    # --------------------------------------------------------

    Dense(
        num_classes,
        activation="softmax"
    )

])


# ============================================================
# 8. COMPILE MODEL
# ============================================================

model.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

print("\nCNN model compiled successfully!")


# ============================================================
# 9. DISPLAY MODEL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("CNN MODEL ARCHITECTURE")
print("=" * 70)

model.summary()


# ============================================================
# 10. TEST MODEL WITH ONE SAMPLE
# ============================================================

print("\n" + "=" * 70)
print("TESTING MODEL OUTPUT")
print("=" * 70)

test_sample = X_train_cnn[:1]

prediction = model.predict(
    test_sample,
    verbose=0
)

print("\nInput Shape:")
print(test_sample.shape)

print("\nOutput Shape:")
print(prediction.shape)

print("\nPrediction Probability Sum:")
print(np.sum(prediction))

print("\nCNN model successfully accepts IQ signal input.")


# ============================================================
# 11. SAVE MODEL ARCHITECTURE
# ============================================================

model_path = os.path.join(
    models_dir,
    "cnn_modulation_classifier.keras"
)

model.save(model_path)

print("\nModel architecture saved:")
print(os.path.abspath(model_path))


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("CNN MODEL DEVELOPMENT COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"\nInput Shape: {X_train_cnn.shape[1:]}")
print(f"Output Classes: {num_classes}")

print("\nCompleted Tasks:")

print("[✓] Processed IQ data loaded")
print("[✓] Input reshaped for CNN")
print("[✓] Labels converted to categorical format")
print("[✓] CNN architecture created")
print("[✓] CNN model compiled")
print("[✓] Model input tested")
print("[✓] Model architecture saved")

print("\nNEXT STAGE: MODEL TRAINING")